# Pipeline 10: Social Media → Donation-Driven Content Decisions

**Organization:** River of Life / Lighthouse Sanctuary (INTEX)  
**Methodology:** CRISP-DM–aligned (see `pipeline_guide.md`)

---

## Executive Summary

This pipeline supports **fundraising-focused social strategy**: it estimates which **planned post characteristics** are associated with **donation-attributed outcomes** recorded in `social_media_posts` (`donation_referrals`, `estimated_donation_value_php`), and scores **draft-style inputs** (platform, format, topic, caption signals, schedule, boost plan) **without** using post-publication vanity metrics as predictors.

**Deliverables**
- **Binary models:** probability that a post will achieve **any attributed donation traction** (organization’s own attribution fields).
- **Regression (optional track):** magnitude of `estimated_donation_value_php` on a log scale for ranking “expected PHP lift.”
- **Explanatory:** logistic / ridge-style coefficients and forest importances on the **same pre-post feature space**.
- **Honest framing:** `donations.csv` contains **`referral_post_id`** for a subset of gifts—useful for **spot checks**, but the main target remains the **aggregated post-level fields** supplied for this case.

*Readers:* Executive Summary → **Problem Framing** → **Donation success vs vanity engagement** → **Recommendations**.

---



## 1. Problem Framing

### Business problem
The organization depends on social channels for visibility but **cannot tell** which creative choices actually **move donations** versus **noise and likes**. Posting feels random; leadership and fundraising want **evidence-based content rules**—what to publish, when, and with which calls-to-action—so social effort **converts to mission revenue**, not just engagement.

### Stakeholders
| Role | Interest |
|------|----------|
| **Leadership / founders** | Strategic narrative, budget justification |
| **Fundraising** | Campaign ROI, donor pipeline |
| **Marketing / outreach** | Editorial calendar, creative briefs, A/B tests |

### Decision supported
- **Content:** which **post types**, **topics**, **CTAs**, and **caption patterns** to prioritize for fundraising posts.
- **Timing:** **day-of-week** and **hour** choices when scheduling.
- **Paid support:** whether a **boost budget** is part of the plan (known before publish).
- **Draft scoring:** estimated **P(donation-attributed success)** from **plannable fields only**—not from likes or reach after the fact.

### Predictive vs explanatory
| Goal | Question | Model role |
|------|----------|------------|
| **Predictive** | Before we publish, how likely is this post to show **donation traction** in our records? | **Random Forest** (+ dummy baseline); strong ranking / nonlinear fit. |
| **Explanatory** | Which **attributes co-occur** with donation outcomes for training and ethics conversations? | **Logistic regression** (balanced): signed coefficients on one-hot + scaled numerics. |

### Target variables (organization-attributed)
**Primary binary — `y_any_donation`:**  
`1` if `donation_referrals ≥ 1` (equivalent in this extract to `estimated_donation_value_php > 0`). These fields are **Lighthouse’s own attribution**, not independent third-party tracking.

**Secondary binary — `y_high_value`:**  
`1` if `estimated_donation_value_php` is in the **top quartile** across all posts—highlights **standout fundraising posts**.

**Regression — `log1p(estimated_donation_value_php)`:**  
Interprets **magnitude**; reported MAE is transformed back where noted.

**Why not “likes” as the target?**  
Likes are **vanity** relative to mission; they appear only in the **diagnostic** section comparing **engagement vs donation outcomes**.

### Success metrics
- **Classification:** **ROC-AUC** (ranking draft posts), **precision / recall / F1** at **operating thresholds** (0.3 / 0.5 / 0.7).
- **Regression:** **MAE / R²** on log scale; business read in approximate **PHP** terms on holdout.

### Limitations (preview)
Attribution is **internal**; **platform algorithms**, **seasonality**, and **unobserved campaigns** confound causal claims—see Section 7.

---



## Data Validity & Leakage Check

### Tables
| Table | Role |
|-------|------|
| **`social_media_posts.csv`** | Primary: **targets** (`donation_referrals`, `estimated_donation_value_php`) and **metadata** / caption / scheduling / boost fields. |
| **`donations.csv` (optional check)** | **`referral_post_id`** links **some** gifts to `post_id`; sparse but useful for **sanity checks**. Not required to train the main model. |

### Pre-post vs post-performance (critical)
**Allowed as model inputs (known when scheduling a post):**  
platform, post type, media type, topic, sentiment label, CTA flags/types, caption text derivatives (length, donate URL, urgency keywords), hashtag/mention counts, **scheduled** `day_of_week` / `post_hour` / month, `is_boosted`, `boost_budget_php`, `features_resident_story`, `campaign_name`.

**Excluded from prediction (would be unknown for a true pre-publish score OR are outcomes):**  
`impressions`, `reach`, `likes`, `comments`, `shares`, `saves`, `click_throughs`, `video_views`, `engagement_rate`, `profile_visits`, `follower_count_at_post`, `subscriber_count_at_post`, `watch_time_seconds`, `avg_view_duration_seconds`, `forwards`, **`donation_referrals`**, **`estimated_donation_value_php`**.

**EDA only:** post-performance fields may appear in **Section 3.5 (Vanity vs donation)** to show **why engagement ≠ fundraising**.

### Train/test
**Stratified holdout** and **cross-validation** with **pipeline-fitted preprocessing** (no global imputation before split).

---



## 2. Data Acquisition & Preparation

### Load & enrich
We parse `created_at` for **month**; derive **lightweight text flags** from `caption` (regex / keywords—explainable, no heavy NLP stack).

### Engineered pre-post features (documentation)
| Feature | Description | Fundraising rationale |
|---------|-------------|----------------------|
| `post_hour`, `day_of_week`, `post_month` | Schedule | Audience online patterns; operational planning. |
| `caption_length`, `word_count` | Copy length | Very short vs long asks may differ in clarity. |
| `num_hashtags`, `mentions_count` | Social mechanics | Discovery vs clutter. |
| `has_call_to_action`, `call_to_action_type` | CTA design | **DonateNow** vs **LearnMore** may differ in conversion intent. |
| `has_donate_url` | Caption links to donate path | Direct giving friction reducer. |
| `has_urgency_word` | “urgent”, “running low”, etc. | Urgency framing—**use ethically**; test empirically. |
| `has_impact_story_lang` | survivor / support / hope lexicon | Storytelling tied to mission identity. |
| `is_fundraising_type` | `post_type == FundraisingAppeal` | Explicit fundraising format. |
| `is_weekend` | Sat/Sun flag | Staffing and audience availability. |
| `is_boosted`, `boost_budget_php` | Paid plan | Reach investment (planned). |
| `features_resident_story` | Flag for resident narrative | Humanizing content hypothesis. |
| `platform`, `media_type`, `content_topic`, `sentiment_tone`, `campaign_name` | Channel & creative context | Platform norms and campaign arcs. |

### Preprocessing
**Median imputation + scaling** for numerics inside **`Pipeline`**; **one-hot** categoricals. Statistics **never** fit on the full dataset before splitting.

---



In [ ]:
import json
import warnings
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore", category=UserWarning)
RANDOM_STATE = 42
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_context("talk", font_scale=0.85)


def resolve_data_and_artifacts() -> tuple[Path, Path]:
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        ml = p / "machine_learning" / "lighthouse_csv_v7"
        if ml.is_dir():
            return ml, p / "machine_learning" / "ml_pipelines" / "artifacts"
        root = p / "lighthouse_csv_v7"
        if root.is_dir():
            return root, p / "ml_pipelines" / "artifacts"
    raise FileNotFoundError("lighthouse_csv_v7 not found.")


DATA_DIR, OUTPUT_DIR = resolve_data_and_artifacts()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
_base = OUTPUT_DIR.parent.parent
print("DATA_DIR:", DATA_DIR.resolve().relative_to(_base.resolve()))
print("OUTPUT_DIR:", OUTPUT_DIR.resolve().relative_to(_base.resolve()))



In [ ]:
def enrich_social_posts(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    d["created_at"] = pd.to_datetime(d["created_at"], errors="coerce")
    cap = d["caption"].fillna("").str.lower()

    d["has_donate_url"] = cap.str.contains(r"lighthouse\.ph/donate|/donate\?", regex=True).astype(int)
    d["has_urgency_word"] = cap.str.contains(
        r"urgent|emergency|running low|reserve is gone|help today|hungry girls|need help now", regex=True
    ).astype(int)
    d["has_impact_story_lang"] = cap.str.contains(
        r"survivor|safehouse|traffick|your support|every (peso|donation)|because of you|graduated|what your support|making possible",
        regex=True,
    ).astype(int)
    d["word_count"] = cap.str.split().str.len().fillna(0).clip(0, 600).astype(int)

    d["is_weekend"] = d["day_of_week"].isin(["Saturday", "Sunday"]).astype(int)
    d["is_fundraising_type"] = (d["post_type"].fillna("") == "FundraisingAppeal").astype(int)
    d["cta_donate_now"] = (d["call_to_action_type"].fillna("") == "DonateNow").astype(int)
    d["post_month"] = d["created_at"].dt.month.astype(str)

    for c in ["is_boosted", "has_call_to_action", "features_resident_story"]:
        if c in d.columns:
            d[c] = (
                d[c]
                .astype(str)
                .str.lower()
                .map({"true": 1, "false": 0, "1": 1, "0": 0})
                .fillna(0)
                .astype(int)
            )

    for c in ["post_hour", "caption_length", "num_hashtags", "mentions_count", "boost_budget_php"]:
        if c in d.columns:
            d[c] = pd.to_numeric(d[c], errors="coerce")

    return d


soc = pd.read_csv(DATA_DIR / "social_media_posts.csv", parse_dates=["created_at"])
soc = soc.dropna(subset=["donation_referrals", "estimated_donation_value_php"]).copy()
soc = enrich_social_posts(soc)

soc["donation_referrals"] = pd.to_numeric(soc["donation_referrals"], errors="coerce").fillna(0)
soc["estimated_donation_value_php"] = pd.to_numeric(soc["estimated_donation_value_php"], errors="coerce").fillna(0)

soc["y_any_donation"] = (soc["donation_referrals"] >= 1).astype(int)
q75_val = soc["estimated_donation_value_php"].quantile(0.75)
soc["y_high_value"] = (soc["estimated_donation_value_php"] >= q75_val).astype(int)
soc["y_log_value"] = np.log1p(soc["estimated_donation_value_php"])

# Optional: linkage sanity check
try:
    don = pd.read_csv(DATA_DIR / "donations.csv")
    n_ref = don["referral_post_id"].notna().sum()
    print("Donations with referral_post_id (for external validation):", int(n_ref))
except FileNotFoundError:
    print("(donations.csv not found for linkage check)")

print("Posts:", len(soc), "  P(any donation attribution):", round(soc["y_any_donation"].mean(), 3))
print("Top-quartile value threshold (PHP):", round(q75_val, 2))
soc.head(3)



## 3. Exploratory Data Analysis

Charts use the **full historical table** for **marginal patterns**. The **predictive model** uses only **pre-post** columns. **Section 3.5** deliberately overlays **post-hoc engagement** to contrast **vanity vs donation** outcomes.

---



In [ ]:
eda = soc.copy()

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
vc = eda["y_any_donation"].value_counts().reindex([0, 1], fill_value=0)
ax[0].bar(["No attributed\ndonations (0)", "≥1 referral /\nvalue>0 (1)"], vc.values, color=["#abd9e9", "#d73027"])
ax[0].set_ylabel("Posts")
ax[0].set_title("Target: any donation-attributed outcome")
eda["y_any_donation"].value_counts(normalize=True).plot(kind="bar", ax=ax[1], color=["#abd9e9", "#d73027"], rot=0)
ax[1].set_title("Class balance")
ax[1].set_ylabel("Share")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
by_pl = eda.groupby("platform")["y_any_donation"].agg(["mean", "count"]).sort_values("mean", ascending=False)
plt.bar(by_pl.index.astype(str), by_pl["mean"], color="#4575b4")
plt.ylabel("P(any donation attribution)")
plt.title("Donation-attributed success rate by platform")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 4))
by_pt = eda.groupby("post_type")["y_any_donation"].mean().sort_values(ascending=False)
by_pt.plot(kind="bar", color="#d95f02", rot=35)
plt.ylabel("P(any donation attribution)")
plt.title("Success rate by post_type")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
eda.groupby("day_of_week")["y_any_donation"].mean().reindex(
    ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
).plot(kind="bar", ax=ax[0], color="#1b7837", rot=30)
ax[0].set_title("By day of week")
ax[0].set_ylabel("P(success)")
eda["post_hour_binned"] = pd.cut(eda["post_hour"].fillna(12), bins=[-1, 6, 12, 18, 24], labels=["night", "morning", "afternoon", "evening"])
eda.groupby("post_hour_binned", observed=True)["y_any_donation"].mean().plot(kind="bar", ax=ax[1], color="#762a83", rot=0)
ax[1].set_title("By time of day (binned)")
ax[1].set_ylabel("P(success)")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.boxplot(data=eda, x="y_any_donation", y="caption_length", ax=ax[0])
ax[0].set_title("Caption length vs success")
sns.boxplot(data=eda, x="y_any_donation", y="word_count", ax=ax[1])
ax[1].set_title("Word count vs success")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(1, 3, figsize=(13, 4))
for a, col, title in zip(
    ax,
    ["has_call_to_action", "has_donate_url", "cta_donate_now"],
    ["Has any CTA", "Donate URL in caption", "CTA type = DonateNow"],
):
    eda.groupby(col)["y_any_donation"].mean().plot(kind="bar", ax=a, color="#e0f3f8")
    a.set_title(title)
    a.set_ylabel("P(success)")
    a.set_xticklabels(a.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()

top = eda.nlargest(15, "estimated_donation_value_php")[
    ["platform", "post_type", "call_to_action_type", "has_donate_url", "estimated_donation_value_php", "donation_referrals"]
]
bot = eda.nsmallest(15, "estimated_donation_value_php")[
    ["platform", "post_type", "call_to_action_type", "has_donate_url", "estimated_donation_value_php", "donation_referrals"]
]
print("Top 15 posts by attributed PHP (summary)")
print(top.groupby(["platform", "post_type"]).size().sort_values(ascending=False).head(8))
print("\nBottom 15: common patterns")
print(bot["post_type"].value_counts().head())

miss = eda[
    ["caption_length", "boost_budget_php", "num_hashtags", "call_to_action_type"]
].isna().mean()
plt.figure(figsize=(7, 3))
miss.sort_values().plot(kind="barh", color="#a65628")
plt.title("Missingness (pre-engineered columns)")
plt.tight_layout()
plt.show()

num_for_corr = [
    "post_hour",
    "caption_length",
    "word_count",
    "num_hashtags",
    "mentions_count",
    "is_boosted",
    "boost_budget_php",
    "has_donate_url",
    "has_urgency_word",
    "has_impact_story_lang",
    "y_any_donation",
]
cm = eda[num_for_corr].corr(numeric_only=True, min_periods=30)
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt=".2f", cmap="vlag", center=0, annot_kws={"size": 8})
plt.title("Correlation: pre-post numeric features + target")
plt.tight_layout()
plt.show()



### EDA interpretations

**Target distribution**  
Roughly **one-third** of posts show **no** attributed donation activity in this dataset; the rest have **some** referral/value recorded. Models must handle **moderate imbalance**; **ROC-AUC** and **threshold tuning** matter more than raw accuracy.

**Platform**  
If some platforms show **higher** attributed-success rates, the model will **encode** that pattern—but **confounding** is likely (content mix differs by channel). Use findings to **hypothesize channel-specific playbooks**, not to **abandon** weaker platforms without experiments.

**Post type**  
**FundraisingAppeal** and similar types often separate from **purely educational** posts; this informed **`is_fundraising_type`** and keeping **`post_type`** as a **categorical** feature.

**Timing**  
Day-of-week and **time-of-day** bins show whether **scheduling** might nudge outcomes; effects are usually **small** but **cheap** to test operationally.

**Caption length / word count**  
Boxplots show whether **concise vs detailed** copy aligns with success in **this** history—inform **editorial guidelines**, not universal truth.

**CTA and donate URL**  
Bars compare **explicit asks** and **deep links**. If **DonateNow** or **donate URLs** associate with higher success, prioritize them in **fundraising calendar**—while noting **selection bias** (fundraisers use CTAs more when they intend to raise).

**Top vs bottom posts**  
Summaries show **concentration** of high PHP in a few **platform × format** combos; bottom posts are often **non-fundraising** types—another reason the target is **attribution-based**, not “all posts should fundraise.”

**Missingness**  
Sparse **boost_budget** or missing CTA type should be **imputed in-pipeline**; monitor if **missingness** itself correlates with outcomes (documentation practice).

**Heatmap**  
Pre-post features show **modest** linear correlation with **`y_any_donation`**; **tree models** remain useful for **interactions** (e.g., urgency × platform).

---



## 3.5 Donation Success vs Vanity Engagement

**Why engagement is not our target**  
**Likes, reach, and engagement_rate** can rise for **controversial** or **entertaining** content that **does not** produce gifts. For a nonprofit, **fundraising effectiveness** is the strategic outcome—engagement is at best a **leading indicator** and at worst a **distraction**.

**What we do here**  
We **do not** feed engagement into the **predictive** model. Below, we **only compare** correlations / simple visuals between **post-hoc engagement_rate** and **donation_referrals** to show **divergence**.

**Proxy target honesty**  
`donation_referrals` / `estimated_donation_value_php` are **internal attributions**, not bank reconciliation. They are still **closer to mission** than likes. **`donations.referral_post_id`** partially corroborates linkage for a subset of rows.

**Residual risk**  
Staff may **attribute** more generously to posts they remember; **campaign spikes** create **confounding**. Treat scores as **decision support**, not accounting.

---



In [ ]:
if "engagement_rate" in soc.columns:
    er = pd.to_numeric(soc["engagement_rate"], errors="coerce")
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    sns.scatterplot(x=er, y=soc["donation_referrals"], alpha=0.35, ax=ax[0])
    ax[0].set_title("Engagement rate vs donation referrals (post-hoc)")
    sns.scatterplot(x=er, y=soc["estimated_donation_value_php"], alpha=0.35, ax=ax[1])
    ax[1].set_title("Engagement rate vs attributed PHP (post-hoc)")
    plt.tight_layout()
    plt.show()
    sub = pd.DataFrame({"engagement_rate": er, "y_any_donation": soc["y_any_donation"]}).dropna()
    print(
        "Correlation engagement_rate vs y_any_donation:",
        round(sub["engagement_rate"].corr(sub["y_any_donation"]), 4),
    )
else:
    print("No engagement_rate column — skip vanity comparison plots.")



## 4. Modeling & Feature Selection

### Pre-post feature list (no post-performance leakage)
**Numeric:** `post_hour`, `caption_length`, `word_count`, `num_hashtags`, `mentions_count`, `is_boosted`, `boost_budget_php`, `has_call_to_action`, `features_resident_story`, `has_donate_url`, `has_urgency_word`, `has_impact_story_lang`, `is_weekend`, `cta_donate_now`, `is_fundraising_type`

**Categorical:** `platform`, `post_type`, `media_type`, `content_topic`, `sentiment_tone`, `call_to_action_type`, `campaign_name`, `day_of_week`, `post_month`

### Models
| Model | Role |
|-------|------|
| **Dummy (stratified)** | Baseline — must beat for usefulness. |
| **Logistic (balanced)** | **Explanatory** — coefficients. |
| **Random Forest** | **Predictive** — nonlinearities / interactions. |

### Validation
**StratifiedKFold (5)** on training rows + **20% holdout** (stratified) for threshold reporting. **All imputation/scaling inside `Pipeline`.**

---



In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    classification_report,
    mean_absolute_error,
    precision_recall_fscore_support,
    r2_score,
    roc_auc_score,
)
from sklearn.model_selection import KFold, StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.base import clone

FEATURE_NUM = [
    "post_hour",
    "caption_length",
    "word_count",
    "num_hashtags",
    "mentions_count",
    "is_boosted",
    "boost_budget_php",
    "has_call_to_action",
    "features_resident_story",
    "has_donate_url",
    "has_urgency_word",
    "has_impact_story_lang",
    "is_weekend",
    "cta_donate_now",
    "is_fundraising_type",
]
FEATURE_CAT = [
    "platform",
    "post_type",
    "media_type",
    "content_topic",
    "sentiment_tone",
    "call_to_action_type",
    "campaign_name",
    "day_of_week",
    "post_month",
]

X = soc[FEATURE_NUM + FEATURE_CAT]
y_cls = soc["y_any_donation"]
y_reg = soc["y_log_value"]

_num = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]
)
preprocess = ColumnTransformer(
    [
        ("num", _num, FEATURE_NUM),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), FEATURE_CAT),
    ]
)


def clf_pipe(est):
    return Pipeline([("prep", preprocess), ("clf", est)])


def reg_pipe(est):
    return Pipeline([("prep", preprocess), ("model", est)])


X_tr, X_te, y_tr, y_te, yreg_tr, yreg_te = train_test_split(
    X, y_cls, y_reg, test_size=0.2, random_state=RANDOM_STATE, stratify=y_cls
)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

clf_models = {
    "dummy_stratified": DummyClassifier(strategy="stratified", random_state=RANDOM_STATE),
    "logistic_balanced": LogisticRegression(max_iter=5000, class_weight="balanced", random_state=RANDOM_STATE, C=1.0),
    "random_forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        min_samples_leaf=4,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
}

rows_cv = []
for name, est in clf_models.items():
    pipe = clf_pipe(est)
    sc = cross_validate(
        pipe,
        X_tr,
        y_tr,
        cv=skf,
        scoring=["roc_auc", "f1", "precision", "recall"],
        n_jobs=-1,
    )
    rows_cv.append(
        {
            "model": name,
            "roc_auc_mean": sc["test_roc_auc"].mean(),
            "roc_auc_std": sc["test_roc_auc"].std(),
            "f1_mean": sc["test_f1"].mean(),
            "precision_mean": sc["test_precision"].mean(),
            "recall_mean": sc["test_recall"].mean(),
        }
    )

cv_df = pd.DataFrame(rows_cv).set_index("model")
print("Classification CV (train split only, 5-fold stratified):")
print(cv_df.round(4).to_string())

reg_models = {
    "dummy_mean": DummyRegressor(strategy="mean"),
    "ridge": Ridge(alpha=2.0),
    "rf_reg": RandomForestRegressor(
        n_estimators=300,
        max_depth=10,
        min_samples_leaf=4,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
}
kf_reg = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
rows_r = []
for name, est in reg_models.items():
    pipe = reg_pipe(est)
    sc = cross_validate(
        pipe,
        X_tr,
        yreg_tr,
        cv=kf_reg,
        scoring={"mae": "neg_mean_absolute_error", "r2": "r2"},
        n_jobs=-1,
    )
    rows_r.append(
        {
            "model": name,
            "mae_log_mean": -sc["test_mae"].mean(),
            "mae_log_std": sc["test_mae"].std(),
            "r2_mean": sc["test_r2"].mean(),
        }
    )
print("\nRegression CV on log1p(PHP):")
print(pd.DataFrame(rows_r).set_index("model").round(4).to_string())



In [ ]:
best_clf = clf_pipe(clf_models["random_forest"])
best_clf.fit(X_tr, y_tr)
p_rf = best_clf.predict_proba(X_te)[:, 1]

log_fit = clf_pipe(clf_models["logistic_balanced"])
log_fit.fit(X_tr, y_tr)
p_log = log_fit.predict_proba(X_te)[:, 1]

print("Held-out ROC-AUC — RF:", round(roc_auc_score(y_te, p_rf), 4))
print("Held-out ROC-AUC — Logistic:", round(roc_auc_score(y_te, p_log), 4))

thresholds = [0.3, 0.5, 0.7]
th_rows = []
for t in thresholds:
    pred = (p_rf >= t).astype(int)
    prec, rec, f1, _ = precision_recall_fscore_support(y_te, pred, average="binary", zero_division=0)
    th_rows.append(
        {"threshold": t, "flagged_share": pred.mean(), "precision": prec, "recall": rec, "f1": f1}
    )
print("\nRandom Forest — threshold tradeoffs (test):")
print(pd.DataFrame(th_rows).round(4).to_string(index=False))

reg_best = reg_pipe(reg_models["rf_reg"])
reg_best.fit(X_tr, yreg_tr)
pred_log = reg_best.predict(X_te)
pred_php = np.expm1(pred_log)
true_php = np.expm1(yreg_te)
print("\nRegression holdout MAE (PHP space, approximate):", round(mean_absolute_error(true_php, pred_php), 2))
print("R2 log-scale:", round(r2_score(yreg_te, pred_log), 4))



In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay

ConfusionMatrixDisplay.from_predictions(y_te, (p_rf >= 0.5).astype(int), ax=ax[0])
ax[0].set_title("RF confusion @ 0.5")
RocCurveDisplay.from_predictions(y_te, p_rf, ax=ax[1])
ax[1].set_title("ROC — Random Forest (holdout)")
plt.tight_layout()
plt.show()



## 5. Evaluation & Interpretation

### Reading the CV table
**Mean ± implicit std** from folds: compare **ROC-AUC** vs **dummy**. If **AUC ≈ 0.5**, the pre-post fields carry **little** linearly separable signal and the tool should **not** drive major budget shifts until more **labeled** posts accumulate.

### Holdout and thresholds
**Flagged share** at 0.3 / 0.5 / 0.7 shows how many drafts would be treated as “expected winners.” **Lower threshold** → **higher recall** (fewer missed strong fundraisers) but **more false positives** (staff prioritizes a post that underperforms).

### Business translation
| Error type | Meaning | Mitigation |
|------------|---------|------------|
| **False positive** | Model expects donation traction; post underperforms | Treat as **hypothesis**; run **paired** creative tests |
| **False negative** | Model is pessimistic; post would have done well | **Rotating** exploratory slots so not only “predicted winners” air |

### Regression track
**MAE in PHP space** is approximate (back-transform of log). Use regression primarily to **rank** planned posts by **expected scale**, not exact peso forecasting.

---



## Business Interpretation (plain language)

Scores answer: *“Given how we plan this post (channel, format, copy signals, schedule, boost), does it resemble posts that previously earned attributed donations in our CRM-style export?”*

**Reliability:** moderate—social algorithms and campaigns shift. **Retrain quarterly** and **A/B** major format changes.

**Use in workflow:** **content calendar triage** + **creative brief checklist** (CTA, donate link, storytelling flags)—**not** public-facing scores.

---



## 6. Feature Importance & Actionable Insights

The next cell shows **logistic coefficients** (directional associations) and **random forest importances** (split-based contribution). They can **disagree**—use both.

---



In [ ]:
fn = log_fit.named_steps["prep"].get_feature_names_out()
coefs = pd.Series(log_fit.named_steps["clf"].coef_.ravel(), index=fn).sort_values()

plt.figure(figsize=(8, max(6, len(coefs) * 0.12)))
coefs.tail(18).plot(kind="barh", color=np.where(coefs.tail(18).values < 0, "#1a9850", "#d73027"))
plt.title("Logistic: higher positive → higher log-odds of donation attribution")
plt.tight_layout()
plt.show()

imp = pd.Series(best_clf.named_steps["clf"].feature_importances_, index=fn).sort_values(ascending=False)
plt.figure(figsize=(8, 5))
imp.head(18).iloc[::-1].plot(kind="barh", color="#4575b4")
plt.title("Random Forest: top feature importances")
plt.tight_layout()
plt.show()

print("Top 12 importances:\n", imp.head(12).round(4).to_string())



### Recommendations (concrete)

1. **Fundraising posts:** Prefer **`FundraisingAppeal`** (or equivalent) when the objective is gifts; pair with **`DonateNow`** CTA and a **`lighthouse.ph/donate`** link when tone allows—**test** against softer CTAs for donor fatigue.
2. **Storytelling:** If **`has_impact_story_lang`** or **`features_resident_story`** rank highly, **protect dignity**—use **aggregated / consent-reviewed** story patterns in briefs, not exploitative tropes.
3. **Urgency language:** If urgency flags associate with outcomes, **coordinate with compliance/ethics**; never fabricate crises; measure **long-term donor trust**.
4. **Platform mix:** If one platform dominates importances, **maintain** presence elsewhere for **diversification**, but **weight production** toward historically attributed channels **until experiments** say otherwise.
5. **Timing:** If **day_of_week** / **post_hour** matter modestly, run **cheap schedule tests** (same creative, two slots).
6. **Boost budget:** If **boost_budget_php** matters, align **paid spend** with **high-confidence fundraising creative**, not generic awareness posts.

---



## 7. Causal & Relationship Analysis

### What relationships appear?
Strong models usually emphasize **fundraising-oriented formats**, **explicit CTAs**, **donate URLs**, and **platform/context** dummies. **Urgency** and **story** language may appear if historically paired with appeals.

### Plausibly actionable
**Creative standards** (CTA + link + format), **editorial templates**, and **channel-specific** playbooks—implemented through **pilots** with **tracked** donation fields.

### Not causal
**Correlation ≠ causation.** Higher **boost budgets** may **cause** reach—or **staff may boost** posts they already believe will win. **Platform** effects confound **audience** and **content mix**. **Seasonal campaigns** drive both topics and gifts.

### Limits
We do **not** observe **true** counterfactuals (what if the same post aired elsewhere). **Algorithm changes** and **external shocks** break historical patterns.

### Value despite limits
The pipeline still supports **transparent prioritization**, **training**, and **hypothesis generation** for **controlled experiments**—the honest path from analytics to **learning**.

---



## Key Findings

- **Donation-attributed fields** anchor the target; **engagement** is diagnostic only.
- **Pre-post-only** features support **draft scoring** without leakage from likes/reach.
- **RF vs logistic** pair **prediction** with **explanation**; **thresholds** must match **team capacity** for “priority posts.”

---



## 8. Deployment Notes (brief)

- **Batch score** upcoming posts from the **planner UI**; store `p_donation_success`, optional `expected_log_value`, and **top 3** driver tags (plain language).
- **Retrain** after major campaigns or **platform** changes.
- **Legal/comms** review for **storytelling** and **urgency** ethics.

**Artifact:** joblib bundle with **fitted pipelines** and feature lists.

---



In [ ]:
from joblib import dump

bundle = {
    "clf_rf": clone(best_clf).fit(X, y_cls),
    "clf_logit": clone(log_fit).fit(X, y_cls),
    "reg_rf": clone(reg_best).fit(X, y_reg),
    "feature_num": FEATURE_NUM,
    "feature_cat": FEATURE_CAT,
    "target_any_donation": "y_any_donation (donation_referrals>=1)",
    "value_transform": "log1p(estimated_donation_value_php)",
}
dump(bundle, OUTPUT_DIR / "social_donation_impact_v4.joblib")
print("Saved", OUTPUT_DIR / "social_donation_impact_v4.joblib")

# Minimal handoff output contract
scored_posts = soc.copy()
scored_posts["predicted_donation_value_php"] = np.expm1(bundle["reg_rf"].predict(X))
social_preview = scored_posts[["post_id", "platform", "post_type", "predicted_donation_value_php"]].sort_values(
    "predicted_donation_value_php", ascending=False
)
social_preview.head(15)



---

## Stakeholder summary (Pipeline 10: Social media → donation-driven content)

**Purpose:** Align **social strategy with fundraising**, not vanity metrics: score **planned post traits** (platform, format, topic, CTA, caption cues like donate links and story/urgency language, schedule, boost plan) against **organization-attributed donation outcomes**. **Likes and reach are not used as predictors**; they appear only in the vanity-vs-donation comparison.

**What the model does:** **Classification** for probability of **any attributed donation traction**; **regression** on **log PHP** for rough scale ranking. **Random forest** for predictive strength, **logistic regression** for interpretable drivers, **dummy baseline** to prove lift over chance—all with **pipeline preprocessing** and **threshold tradeoffs** for operations.

**Why it matters:** Reduces **random posting**; gives fundraising and comms a shared, **evidence-based** lens for **what to test** (asks, formats, timing) tied to **mission revenue**.

**Explanatory angle:** Drivers inform **creative briefs** with explicit limits: **platform, campaigns, and boost spend** confound simple causal stories.

**One-line for leadership:** *We score draft posts by choices we make before publish against which past posts actually showed attributed gifts—so we invest in patterns tied to donations, not just likes.*
